<a href="https://colab.research.google.com/github/sam786123-tech/Albert-Einstein/blob/main/Phishing_Email_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
COMPLETE PHISHING EMAIL DETECTOR - All-in-One
Machine Learning model to detect phishing emails
Includes: Training, Testing, Web Interface, and Dataset Generation
"""

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import re
import joblib
import os
import random
from flask import Flask, render_template, request, jsonify
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ============================================================
# PHISHING DETECTOR CLASS
# ============================================================

class PhishingEmailDetector:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
        self.classifier = RandomForestClassifier(n_estimators=100, random_state=42)
        self.is_trained = False

    def extract_features_from_text(self, text):
        """Extract handcrafted features from email text"""
        if not isinstance(text, str):
            text = str(text)

        text_lower = text.lower()

        features = {
            'has_url': 1 if re.search(r'https?://|www\.', text_lower) else 0,
            'url_count': len(re.findall(r'https?://[^\s]+', text_lower)),
            'has_shortened_url': 1 if re.search(r'bit\.ly|tinyurl|goo\.gl|ow\.ly', text_lower) else 0,
            'urgent_words': 1 if re.search(r'urgent|immediately|asap|verify now|action required', text_lower) else 0,
            'account_words': 1 if re.search(r'account|password|verify|update|confirm|security', text_lower) else 0,
            'prize_words': 1 if re.search(r'winner|congratulations|prize|lottery|won', text_lower) else 0,
            'payment_words': 1 if re.search(r'payment|credit card|bank|transfer|invoice', text_lower) else 0,
            'personal_info': 1 if re.search(r'ssn|social security|tax id|driver license', text_lower) else 0,
            'has_attachment': 1 if re.search(r'\.zip|\.exe|\.scr|\.bat', text_lower) else 0,
            'has_many_links': 1 if len(re.findall(r'https?://', text_lower)) > 3 else 0,
            'has_ip_address': 1 if re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', text_lower) else 0,
            'has_many_exclamation': 1 if text_lower.count('!') > 3 else 0,
            'text_length': min(len(text), 10000),
            'digit_ratio': sum(c.isdigit() for c in text) / max(len(text), 1),
            'special_char_ratio': sum(not c.isalnum() and not c.isspace() for c in text) / max(len(text), 1)
        }
        return features

    def create_feature_vector(self, text):
        """Convert text to feature vector"""
        features = self.extract_features_from_text(text)
        return np.array(list(features.values()))

    def generate_dataset(self, num_samples=500):
        """Generate synthetic dataset for training"""
        print("📊 Generating training dataset...")
        data = []

        # Phishing email templates
        phishing_templates = [
            "URGENT: Your {company} account has been suspended! Click here to verify: {url}",
            "Dear Customer, We noticed suspicious activity. Verify now: {url}",
            "CONGRATULATIONS! You've won {prize}! Claim now: {url}",
            "{company} Security Alert: Your password will expire. Update here: {url}",
            "Your account will be closed! Action required: {url}",
            "You have a pending refund of ${amount}. Claim it: {url}",
            "IRS Notice: You owe ${amount} in taxes. Pay immediately: {url}",
            "Someone logged into your account from {location}. Verify: {url}"
        ]

        # Safe email templates
        safe_templates = [
            "Subject: Meeting Tomorrow\nHi team, Reminder about our meeting at {time} in {room}.",
            "Your order #{order} has been confirmed. Delivery in {days} days.",
            "Welcome to our platform! Complete your profile to get started.",
            "Weekly Newsletter: Check out our latest blog post about {topic}.",
            "{sender} shared a document with you on Google Drive: {doc}",
            "Your password was successfully changed. If this wasn't you, contact support.",
            "Thank you for your purchase! Your receipt is attached.",
            "Project Update: {progress} tasks completed this week."
        ]

        companies = ["PayPal", "Netflix", "Amazon", "Bank of America", "Chase", "Apple", "Microsoft", "Google"]
        urls = ["http://bit.ly/verify", "https://tinyurl.com/secure", "http://fake-login.com", "https://verify-account.net"]
        prizes = ["$1,000,000", "iPhone 15", "MacBook Pro", "$500 Gift Card"]
        locations = ["New York", "London", "Tokyo", "Unknown Device", "Russia"]
        topics = ["Cybersecurity", "Machine Learning", "Web Development", "AI"]
        senders = ["John", "Sarah", "Mike", "Lisa", "David"]
        docs = ["Q4_Report.pdf", "Project_Plan.docx", "Budget.xlsx"]
        progresses = ["5/8", "12/15", "3/4", "90%"]

        # Generate phishing emails
        for _ in range(num_samples // 2):
            template = random.choice(phishing_templates)
            email = template.format(
                company=random.choice(companies),
                url=random.choice(urls),
                prize=random.choice(prizes),
                amount=random.randint(100, 5000),
                location=random.choice(locations)
            )
            # Add urgency
            if random.random() > 0.5:
                email = "URGENT! " + email
            data.append({'email_text': email, 'label': 1})

        # Generate safe emails
        for _ in range(num_samples // 2):
            template = random.choice(safe_templates)
            email = template.format(
                time=f"{random.randint(9,17)}:00 AM",
                room=random.choice(["A", "B", "C", "Main Conference"]),
                order=random.randint(10000, 99999),
                days=random.randint(1,7),
                topic=random.choice(topics),
                sender=random.choice(senders),
                doc=random.choice(docs),
                progress=random.choice(progresses)
            )
            data.append({'email_text': email, 'label': 0})

        random.shuffle(data)
        df = pd.DataFrame(data)
        df.to_csv('phishing_dataset.csv', index=False)
        print(f"   ✅ Generated {len(df)} emails ({num_samples//2} phishing, {num_samples//2} safe)")
        return df

    def train_model(self, dataset_file=None):
        """Train the ML model"""
        print("\n" + "="*60)
        print("🔒 PHISHING EMAIL DETECTION - TRAINING MODE")
        print("="*60)

        # Load or generate dataset
        if dataset_file and os.path.exists(dataset_file):
            df = pd.read_csv(dataset_file)
            print(f"✅ Loaded dataset: {len(df)} emails")
        else:
            df = self.generate_dataset(500)

        # Prepare data
        X_text = df['email_text'].values
        y = df['label'].values

        # Extract features
        print("🔄 Extracting features...")
        X_features = np.array([self.create_feature_vector(text) for text in X_text])
        X_tfidf = self.vectorizer.fit_transform(X_text).toarray()
        X_combined = np.hstack([X_features, X_tfidf])

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)
        print(f"📊 Training samples: {len(X_train)} | Testing samples: {len(X_test)}")

        # Train
        print("🔄 Training Random Forest Classifier...")
        self.classifier.fit(X_train, y_train)

        # Evaluate
        y_pred = self.classifier.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)

        print(f"\n📈 RESULTS:")
        print(f"   Accuracy: {accuracy*100:.2f}%")
        print(f"\n   Confusion Matrix:")
        print(f"   ┌─────────────────────────────────────┐")
        print(f"   │              Predicted              │")
        print(f"   │         Safe     Phishing           │")
        print(f"   │ Safe    {cm[0,0]:5d}     {cm[0,1]:5d}        │")
        print(f"   │ Phish   {cm[1,0]:5d}     {cm[1,1]:5d}        │")
        print(f"   └─────────────────────────────────────┘")

        self.is_trained = True

        # Save model
        joblib.dump(self.classifier, 'phishing_model.pkl')
        joblib.dump(self.vectorizer, 'vectorizer.pkl')
        print("\n✅ Model saved to 'phishing_model.pkl'")

        return accuracy, cm

    def predict_email(self, email_text):
        """Predict if email is phishing or safe"""
        if not self.is_trained:
            if os.path.exists('phishing_model.pkl'):
                self.classifier = joblib.load('phishing_model.pkl')
                self.vectorizer = joblib.load('vectorizer.pkl')
                self.is_trained = True
            else:
                raise Exception("Model not trained. Run train_model() first.")

        features = self.create_feature_vector(email_text)
        tfidf = self.vectorizer.transform([email_text]).toarray()
        combined = np.hstack([features.reshape(1, -1), tfidf])

        prediction = self.classifier.predict(combined)[0]
        probability = self.classifier.predict_proba(combined)[0]

        return {
            'is_phishing': bool(prediction == 1),
            'confidence': max(probability) * 100,
            'confidence_safe': probability[0] * 100,
            'confidence_phishing': probability[1] * 100,
            'features': self.extract_features_from_text(email_text)
        }

# ============================================================
# FLASK WEB APPLICATION
# ============================================================

app = Flask(__name__)
detector = PhishingEmailDetector()

# HTML template as string (embedded for single file)
HTML_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>PhishGuard AI | Phishing Email Detector</title>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap" rel="stylesheet">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: 'Inter', sans-serif;
            background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%);
            min-height: 100vh;
            color: #fff;
        }
        .container { max-width: 1200px; margin: 0 auto; padding: 30px 20px; }
        .header { text-align: center; margin-bottom: 40px; }
        .logo { display: flex; align-items: center; justify-content: center; gap: 15px; margin-bottom: 15px; }
        .logo i { font-size: 48px; color: #00d4ff; }
        .logo h1 { font-size: 36px; font-weight: 700; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
        .badge { display: inline-block; padding: 8px 20px; background: rgba(0,212,255,0.1); border: 1px solid rgba(0,212,255,0.3); border-radius: 40px; font-size: 13px; margin: 0 8px; }
        .card { background: rgba(255,255,255,0.05); backdrop-filter: blur(10px); border-radius: 24px; padding: 30px; margin-bottom: 30px; border: 1px solid rgba(255,255,255,0.1); }
        .card h2 { font-size: 22px; margin-bottom: 20px; display: flex; align-items: center; gap: 10px; }
        .email-input { width: 100%; min-height: 200px; padding: 20px; font-size: 15px; font-family: 'Inter', monospace; background: rgba(0,0,0,0.3); border: 2px solid rgba(255,255,255,0.1); border-radius: 16px; color: white; resize: vertical; }
        .email-input:focus { outline: none; border-color: #00d4ff; }
        .btn-analyze { margin-top: 20px; padding: 14px 30px; font-size: 16px; font-weight: 600; border: none; border-radius: 12px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; cursor: pointer; transition: all 0.3s ease; }
        .btn-analyze:hover { transform: translateY(-2px); box-shadow: 0 10px 30px rgba(102,126,234,0.4); }
        .examples { display: flex; gap: 12px; flex-wrap: wrap; margin-top: 15px; }
        .example-btn { padding: 8px 16px; background: rgba(255,255,255,0.1); border: 1px solid rgba(255,255,255,0.2); border-radius: 20px; font-size: 12px; cursor: pointer; transition: all 0.2s ease; }
        .example-btn:hover { background: rgba(255,255,255,0.2); }
        .result-card { border-radius: 20px; padding: 25px; margin-bottom: 20px; }
        .result-phishing { background: linear-gradient(135deg, rgba(220,53,69,0.2), rgba(220,53,69,0.05)); border-left: 4px solid #dc3545; }
        .result-safe { background: linear-gradient(135deg, rgba(40,167,69,0.2), rgba(40,167,69,0.05)); border-left: 4px solid #28a745; }
        .result-title { font-size: 24px; font-weight: 700; margin-bottom: 10px; }
        .confidence-bar { background: rgba(255,255,255,0.1); border-radius: 10px; height: 8px; margin: 15px 0; overflow: hidden; }
        .confidence-fill { height: 100%; border-radius: 10px; transition: width 0.5s ease; }
        .fill-phishing { background: #dc3545; }
        .fill-safe { background: #28a745; }
        .features-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-top: 20px; }
        .feature-item { background: rgba(255,255,255,0.03); padding: 12px; border-radius: 12px; display: flex; justify-content: space-between; }
        .feature-label { color: #aaa; }
        .feature-true { color: #ff6b6b; }
        .feature-false { color: #28a745; }
        .loading { text-align: center; padding: 40px; }
        .spinner { width: 40px; height: 40px; border: 3px solid rgba(255,255,255,0.1); border-top-color: #667eea; border-radius: 50%; animation: spin 1s linear infinite; margin: 0 auto 15px; }
        @keyframes spin { to { transform: rotate(360deg); } }
        .hidden { display: none; }
        .footer { text-align: center; padding: 30px; color: #888; font-size: 13px; }
        @media (max-width: 768px) { .features-grid { grid-template-columns: 1fr; } }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <div class="logo">
                <i class="fas fa-fish"></i>
                <h1>PhishGuard AI</h1>
            </div>
            <div>
                <span class="badge"><i class="fas fa-brain"></i> Machine Learning</span>
                <span class="badge"><i class="fas fa-chart-line"></i> Random Forest</span>
                <span class="badge"><i class="fas fa-shield-alt"></i> Real-time Detection</span>
            </div>
        </div>
        <div class="card">
            <h2><i class="fas fa-envelope"></i> Paste Email Content</h2>
            <textarea id="emailInput" class="email-input" placeholder="Paste the email content here to analyze..."></textarea>
            <button id="analyzeBtn" class="btn-analyze"><i class="fas fa-shield-hog"></i> Analyze Email</button>
            <div class="examples">
                <span style="color: #aaa;">Try examples:</span>
                <button class="example-btn" data-email="phishing1">🔴 Phishing Example</button>
                <button class="example-btn" data-email="safe1">🟢 Safe Example</button>
            </div>
        </div>
        <div id="loading" class="loading hidden"><div class="spinner"></div><p>Analyzing email with AI model...</p></div>
        <div id="result" class="hidden"></div>
        <div class="footer"><p>🔒 Powered by Random Forest Classifier | Trained on 500+ emails</p></div>
    </div>
    <script>
        const emailInput = document.getElementById('emailInput');
        const analyzeBtn = document.getElementById('analyzeBtn');
        const loading = document.getElementById('loading');
        const resultDiv = document.getElementById('result');
        const examples = {
            phishing1: "URGENT! Your PayPal account has been suspended! Click here to verify: http://bit.ly/verify-paypal. Your account will be closed within 24 hours.",
            safe1: "Hi team, Just a reminder about our meeting tomorrow at 10 AM in Conference Room B. Please bring your weekly updates. Best, John"
        };
        document.querySelectorAll('.example-btn').forEach(btn => {
            btn.addEventListener('click', () => {
                const exampleKey = btn.dataset.email;
                if (examples[exampleKey]) { emailInput.value = examples[exampleKey]; analyzeEmail(); }
            });
        });
        analyzeBtn.addEventListener('click', analyzeEmail);
        async function analyzeEmail() {
            const email = emailInput.value.trim();
            if (!email) { alert('Please enter an email to analyze'); return; }
            loading.classList.remove('hidden');
            resultDiv.classList.add('hidden');
            try {
                const response = await fetch('/api/predict', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ email: email })
                });
                const data = await response.json();
                if (data.error) alert('Error: ' + data.error);
                else displayResult(data);
            } catch (error) { alert('Error: ' + error.message); }
            finally { loading.classList.add('hidden'); }
        }
        function displayResult(data) {
            const isPhishing = data.is_phishing;
            const confidence = data.confidence.toFixed(1);
            const features = data.features;
            let html = `<div class="result-card ${isPhishing ? 'result-phishing' : 'result-safe'}">
                <div class="result-title">${isPhishing ? '🔴 PHISHING DETECTED!' : '🟢 SAFE EMAIL'}</div>
                <div style="font-size: 14px; margin-bottom: 10px;">Confidence: ${confidence}%</div>
                <div class="confidence-bar"><div class="confidence-fill ${isPhishing ? 'fill-phishing' : 'fill-safe'}" style="width: ${confidence}%"></div></div>
                ${!isPhishing ? '<div style="margin-top: 15px; color: #aaa;">This email appears legitimate.</div>' : '<div style="margin-top: 15px; color: #ff6b6b;">⚠️ Warning: This shows phishing characteristics!</div>'}
                <div class="features-grid">
                    <div class="feature-item"><span class="feature-label">🔗 Has URL</span><span class="${features.has_url ? 'feature-true' : 'feature-false'}">${features.has_url ? 'Yes' : 'No'}</span></div>
                    <div class="feature-item"><span class="feature-label">⚠️ Urgent Words</span><span class="${features.urgent_words ? 'feature-true' : 'feature-false'}">${features.urgent_words ? 'Yes' : 'No'}</span></div>
                    <div class="feature-item"><span class="feature-label">🔐 Account Related</span><span class="${features.account_words ? 'feature-true' : 'feature-false'}">${features.account_words ? 'Yes' : 'No'}</span></div>
                    <div class="feature-item"><span class="feature-label">🏆 Prize/Winning</span><span class="${features.prize_words ? 'feature-true' : 'feature-false'}">${features.prize_words ? 'Yes' : 'No'}</span></div>
                    <div class="feature-item"><span class="feature-label">💳 Payment Related</span><span class="${features.payment_words ? 'feature-true' : 'feature-false'}">${features.payment_words ? 'Yes' : 'No'}</span></div>
                </div>
            </div>`;
            resultDiv.innerHTML = html;
            resultDiv.classList.remove('hidden');
        }
    </script>
</body>
</html>
'''

@app.route('/')
def index():
    return HTML_TEMPLATE

@app.route('/api/predict', methods=['POST'])
def predict():
    data = request.get_json()
    email_text = data.get('email', '')
    if not email_text:
        return jsonify({'error': 'No email content provided'}), 400
    result = detector.predict_email(email_text)
    return jsonify(result)

# ============================================================
# COMMAND LINE INTERFACE
# ============================================================

def run_cli():
    """Command line interface for testing"""
    print("\n" + "="*60)
    print("🔒 PHISHING EMAIL DETECTOR - COMMAND LINE")
    print("="*60)

    # Train or load model
    if not os.path.exists('phishing_model.pkl'):
        print("Model not found. Training new model...")
        detector.train_model()
    else:
        detector.classifier = joblib.load('phishing_model.pkl')
        detector.vectorizer = joblib.load('vectorizer.pkl')
        detector.is_trained = True
        print("✅ Model loaded successfully!")

    while True:
        print("\n" + "-"*40)
        print("1. Test an email")
        print("2. Exit")
        choice = input("\nEnter choice (1-2): ").strip()

        if choice == '1':
            print("\n📝 Enter email content (type 'END' on new line to finish):")
            lines = []
            while True:
                line = input()
                if line.strip() == 'END':
                    break
                lines.append(line)
            email_text = ' '.join(lines)
            if email_text:
                result = detector.predict_email(email_text)
                print(f"\n📊 Result: {'🔴 PHISHING' if result['is_phishing'] else '🟢 SAFE'}")
                print(f"   Confidence: {result['confidence']:.1f}%")
            else:
                print("❌ No content entered!")
        elif choice == '2':
            print("\n👋 Goodbye!")
            break

# ============================================================
# MAIN ENTRY POINT
# ============================================================

if __name__ == "__main__":
    import sys

    print("\n" + "="*60)
    print("🔒 PHISHGUARD AI - Complete Phishing Email Detector")
    print("="*60)

    # Check command line arguments
    if len(sys.argv) > 1:
        if sys.argv[1] == 'train':
            # Training mode
            detector.train_model()
        elif sys.argv[1] == 'cli':
            # CLI mode
            run_cli()
        elif sys.argv[1] == 'web':
            # Web mode
            print("\n🌐 Starting web server...")
            print("   Open http://localhost:5000 in your browser")
            print("   Press Ctrl+C to stop\n")
            app.run(debug=True, host='0.0.0.0', port=5000)
        else:
            print("\nUsage:")
            print("   python phishing_detector_complete.py train   - Train the model")
            print("   python phishing_detector_complete.py cli     - Command line interface")
            print("   python phishing_detector_complete.py web     - Web interface")
            print("   python phishing_detector_complete.py         - Start web interface (default)")
    else:
        # Default: Train and start web interface
        print("\n🔄 No existing model found. Training first...")
        detector.train_model()
        print("\n🌐 Starting web server...")
        print("   Open http://localhost:5000 in your browser")
        print("   Press Ctrl+C to stop\n")
        app.run(debug=True, host='0.0.0.0', port=5000)


🔒 PHISHGUARD AI - Complete Phishing Email Detector

Usage:
   python phishing_detector_complete.py train   - Train the model
   python phishing_detector_complete.py cli     - Command line interface
   python phishing_detector_complete.py web     - Web interface
   python phishing_detector_complete.py         - Start web interface (default)
